# Statistical Analysis

This notebook starts by investigating anding the distributions of data from NDT7 and Cloudflare AIM. The data is from September 2025. 

The second part is about investigating the predictions for 24-30 November 2025.


In [194]:
import pandas as pd
import numpy as np
import datetime
from scipy.spatial.distance import jensenshannon
from matplotlib import colormaps
from typing import TypedDict, cast

In [195]:
def get_download_and_upload_df(df):
    df_download = df[~df["download_throughput_mbps"].isna()]
    df_upload = df[~df["upload_throughput_mbps"].isna()]
    return df_download, df_upload

ndt7 = pd.read_csv("data/ndt7_sept.csv").dropna(subset=['client_country_code'])

ndt7_download, ndt7_upload = get_download_and_upload_df(ndt7)
cf_data_mean = pd.read_csv("data/cf_mean_sept.csv").dropna(subset=['client_country_code'])
cf_data_median = pd.read_csv("data/cf_median_sept.csv").dropna(subset=['client_country_code'])
cf_data_90th_percentile = pd.read_csv("data/cf_p90_sept.csv").dropna(subset=['client_country_code'])

countries = np.union1d(
    ndt7_download['client_country_code'].unique(),
    cf_data_mean['client_country_code'].unique()
)

download_study_fields = ["packet_loss_rate", "download_throughput_mbps", "download_latency_ms", "download_jitter_ms"]
upload_study_fields = ["packet_loss_rate", "upload_throughput_mbps", "upload_latency_ms", "upload_jitter_ms"]

download_fields_info = {
    "packet_loss_rate": {
        "title": "Packet Loss Rate",
    },
    "download_throughput_mbps": {
        "title": "Download Throughput",
        "measure_unit": "Mbps"
    },
    "download_latency_ms": {
        "title": "Download Latency",
        "measure_unit": "ms"
    },
    "download_jitter_ms": {
        "title": "Download Jitter",
        "measure_unit": "ms"
    }
}

upload_fields_info = {
    "packet_loss_rate": {
        "title": "Packet Loss Rate",
    },
    "upload_throughput_mbps": {
        "title": "Upload Throughput",
        "measure_unit": "Mbps"
    },
    "upload_latency_ms": {
        "title": "Upload Latency",
        "measure_unit": "ms"
    },
    "upload_jitter_ms": {
        "title": "Upload Jitter",
        "measure_unit": "ms"
    }
}

color_for_country = dict(zip(countries, [colormaps['tab20'](i / max(len(countries) - 1, 1)) for i in range(len(countries))]))

In [196]:
def remove_countries_with_few_measurements(ndt7_download, ndt7_upload, cf_data_mean, cf_data_median, cf_data_90th_percentile, threshold=50):
    datasets = {
        "ndt7_download": ndt7_download,
        "ndt7_upload": ndt7_upload,
        "cf_data_mean": cf_data_mean,
        "cf_data_median": cf_data_median,
        "cf_data_90th_percentile": cf_data_90th_percentile,
    }

    countries_to_drop = set()

    print(f"Checking countries with < {threshold} rows:\n")
    for name, df in datasets.items():
        counts = df['client_country_code'].value_counts()
        few_rows = counts[counts < threshold]
        if not few_rows.empty:
            print(f"In {name}:")
            for country, count in few_rows.items():
                print(f"  {country}: {count} rows")
            countries_to_drop.update(few_rows.index)

    countries_array = countries
    if countries_to_drop:
        print(f"\nDropping countries (fewer than 10 rows in any dataset): {sorted(countries_to_drop)}\n")
        for name in datasets:
            datasets[name] = datasets[name][~datasets[name]['client_country_code'].isin(countries_to_drop)]
        countries_array = np.array([c for c in countries if c not in countries_to_drop])
    else:
        print("\nNo countries to drop.\n")

    return countries_array, datasets["ndt7_download"], datasets["ndt7_upload"], datasets["cf_data_mean"], datasets["cf_data_median"], datasets["cf_data_90th_percentile"]

In [197]:
countries, ndt7_download, ndt7_upload, cf_data_mean, cf_data_median, cf_data_90th_percentile = remove_countries_with_few_measurements(ndt7_download, ndt7_upload, cf_data_mean, cf_data_median, cf_data_90th_percentile, threshold=50)
countries = ['US', 'DE', 'AU']

Checking countries with < 50 rows:

In ndt7_download:
  SL: 47 rows
  MT: 46 rows
  SO: 44 rows
  BT: 41 rows
  MH: 39 rows
  BL: 35 rows
  IL: 32 rows
  NI: 27 rows
  ME: 19 rows
  TT: 18 rows
  BB: 17 rows
  FM: 16 rows
  WS: 14 rows
  BA: 14 rows
  RW: 13 rows
  SI: 13 rows
  IS: 12 rows
  SB: 12 rows
  SR: 11 rows
  MD: 11 rows
  CK: 11 rows
  JO: 10 rows
  TO: 9 rows
  LS: 7 rows
  MP: 6 rows
  TL: 5 rows
  GE: 5 rows
  TV: 5 rows
  AZ: 4 rows
  AG: 4 rows
  VC: 3 rows
  IN: 3 rows
  AS: 2 rows
  BH: 2 rows
  AQ: 2 rows
  DM: 2 rows
  GD: 1 rows
  MV: 1 rows
  SJ: 1 rows
  AM: 1 rows
In ndt7_upload:
  GF: 47 rows
  CI: 43 rows
  SL: 40 rows
  BI: 39 rows
  EE: 38 rows
  VU: 37 rows
  MT: 37 rows
  OM: 33 rows
  MH: 32 rows
  KI: 28 rows
  IL: 27 rows
  BT: 26 rows
  BL: 25 rows
  NI: 25 rows
  BB: 14 rows
  TT: 14 rows
  SO: 13 rows
  SB: 12 rows
  RW: 10 rows
  FM: 10 rows
  IS: 9 rows
  BA: 9 rows
  GE: 9 rows
  MD: 9 rows
  CK: 8 rows
  SR: 8 rows
  SI: 7 rows
  ME: 6 rows
  TO

In [198]:
def get_histogram(data, bin_width=None):
    num_bins = 100
    if bin_width is not None:
        num_bins = int((max(data) - min(data)) / bin_width)
    return np.histogram(data,bins=num_bins, density=True)

In [199]:
def compute_jsd(ndt7_df, cf_mean, cf_median, cf_90th, countries, field):
    results = []
    cf_dfs = {
        "mean": cf_mean,
        "median": cf_median,
        "90th_percentile": cf_90th
    }

    for country in countries:
        ndt_data = ndt7_df[ndt7_df['client_country_code'] == country][[field]].dropna()
        counts_ndt, bin_edges_ndt = get_histogram(ndt_data[field])
        bin_width = bin_edges_ndt[1] - bin_edges_ndt[0]

        for label, cf_df in cf_dfs.items():
            cf_data = cf_df[cf_df['client_country_code'] == country][[field]].dropna()
            if cf_data.empty:
                continue

            counts_cf, _ = get_histogram(cf_data[field], bin_width=bin_width)

            max_len = max(len(counts_ndt), len(counts_cf))
            p = np.zeros(max_len)
            q = np.zeros(max_len)
            p[:len(counts_ndt)] = counts_ndt
            q[:len(counts_cf)] = counts_cf

            js = jensenshannon(p, q, base=np.e) ** 2

            results.append({
                "country": country,
                "cf_source": label,
                "JS Divergence": round(js, 4),
            })

    return pd.DataFrame(results)

In [200]:
def compute_jsd_with_median(ndt7_download, ndt7_upload, cf_median, countries, fields):
    results = []

    for country in countries:
        for field in fields:
            if 'download' in field:
                ndt7_df = ndt7_download
            elif 'upload' in field:
                ndt7_df = ndt7_upload
            else:
                ndt7_df = pd.concat([ndt7_download, ndt7_upload], ignore_index=True)
            ndt_data = ndt7_df[ndt7_df['client_country_code'] == country][[field]].dropna()
            counts_ndt, bin_edges_ndt = get_histogram(ndt_data[field])
            bin_width = bin_edges_ndt[1] - bin_edges_ndt[0]

            cf_data = cf_median[cf_median['client_country_code'] == country][[field]].dropna()

            counts_cf, _ = get_histogram(cf_data[field], bin_width=bin_width)

            max_len = max(len(counts_ndt), len(counts_cf))
            p = np.zeros(max_len)
            q = np.zeros(max_len)
            p[:len(counts_ndt)] = counts_ndt
            q[:len(counts_cf)] = counts_cf

            js = jensenshannon(p, q, base=np.e) ** 2

            results.append({
                "field": field,
                "country": country,
                "JS Divergence": round(js, 4),
            })

    return pd.DataFrame(results)

In [201]:
jsd_countries = ['US', 'BR', 'GB', 'JP']

In [202]:
compute_jsd(ndt7_download, cf_data_mean, cf_data_median, cf_data_90th_percentile, jsd_countries, 'download_latency_ms')

,country,cf_source,JS Divergence
0,US,mean,0.0163
1,US,median,0.0426
2,US,90th_percentile,0.0122
3,BR,mean,0.0726
4,BR,median,0.1000
5,BR,90th_percentile,0.0167
6,GB,mean,0.0113
7,GB,median,0.0258
8,GB,90th_percentile,0.0252
9,JP,mean,0.0977


In [203]:
compute_jsd(ndt7_download, cf_data_mean, cf_data_median, cf_data_90th_percentile, jsd_countries, 'download_throughput_mbps')


,country,cf_source,JS Divergence
0,US,mean,0.3475
1,US,median,0.1609
2,US,90th_percentile,0.0646
3,BR,mean,0.3716
4,BR,median,0.1469
5,BR,90th_percentile,0.2011
6,GB,mean,0.4088
7,GB,median,0.1794
8,GB,90th_percentile,0.0335
9,JP,mean,0.3897


In [204]:
compute_jsd(ndt7_upload, cf_data_mean, cf_data_median, cf_data_90th_percentile, jsd_countries, 'upload_latency_ms')


,country,cf_source,JS Divergence
0,US,mean,0.0267
1,US,median,0.0232
2,US,90th_percentile,0.0330
3,BR,mean,0.0690
4,BR,median,0.0768
5,BR,90th_percentile,0.0408
6,GB,mean,0.0123
7,GB,median,0.0128
8,GB,90th_percentile,0.0198
9,JP,mean,0.0299


In [205]:
compute_jsd(ndt7_upload, cf_data_mean, cf_data_median, cf_data_90th_percentile, jsd_countries, 'upload_throughput_mbps')


,country,cf_source,JS Divergence
0,US,mean,0.0586
1,US,median,0.0358
2,US,90th_percentile,0.1218
3,BR,mean,0.1439
4,BR,median,0.1462
5,BR,90th_percentile,0.2157
6,GB,mean,0.0214
7,GB,median,0.0802
8,GB,90th_percentile,0.0641
9,JP,mean,0.1768


In [206]:
compute_jsd_with_median(ndt7_download, ndt7_upload, cf_data_median, jsd_countries, ['download_jitter_ms', 'upload_jitter_ms', 'packet_loss_rate'])

,field,country,JS Divergence
0,download_jitter_ms,US,0.0401
1,upload_jitter_ms,US,0.0228
2,packet_loss_rate,US,0.0253
3,download_jitter_ms,BR,0.0494
4,upload_jitter_ms,BR,0.0266
5,packet_loss_rate,BR,0.0698
6,download_jitter_ms,GB,0.0669
7,upload_jitter_ms,GB,0.0080
8,packet_loss_rate,GB,0.0387
9,download_jitter_ms,JP,0.0661


## Statistical Significance for Predictions

In [207]:
cities = [("New York", "US"), ("Santiago", "CL"), ("Berlin", "DE"), ("Aden", "YE"), ("Yangon", "MM")]
latency_predictions = pd.read_csv("data/city_latency_predictions.csv")
throughput_predictions = pd.read_csv("data/city_throughput_predictions.csv")

In [208]:
class BootstrapResult(TypedDict):
    mae: float
    rmse: float
    mae_dist: np.ndarray
    rmse_dist: np.ndarray


def compute_confidence_interval(data: np.ndarray, confidence_level: float = 95.0) -> tuple[float, float, float]:
    if len(data) == 0:
        raise ValueError("The list of data is empty. Cannot compute confidence interval.")
    if confidence_level <= 0 or confidence_level >= 100:
        raise ValueError("Confidence level must be between 0 and 100.")
    sample_mean = float(np.mean(data))
    lower_percentile = (100 - confidence_level) / 2
    upper_percentile = 100 - lower_percentile
    lower = float(np.percentile(data, lower_percentile))
    upper = float(np.percentile(data, upper_percentile))
    return sample_mean, lower, upper

In [209]:
def compute_bootstrap_distributions(errors: np.ndarray, num_samples: int) -> tuple[np.ndarray, np.ndarray]:
    if len(errors) == 0:
        raise ValueError("The list of errors is empty. Cannot perform bootstrap.")
    samples = np.random.choice(errors, size=(num_samples, len(errors)), replace=True)
    mae = np.mean(np.abs(samples), axis=1)
    rmse = np.sqrt(np.mean(samples**2, axis=1)) 
    return mae, rmse


def compute_daily_bootstrap_metrics(
        dataset: pd.DataFrame,
        city_name: str,
        country_code: str,
        num_samples = 10000
    ) -> dict[str, BootstrapResult]:
    df = dataset[(dataset['city'] == city_name) & (dataset['country'] == country_code)].copy()

    daily_errors: dict[str, BootstrapResult] = {}
    for date, group in df.groupby('date'):
        errors = group['error'].to_numpy(dtype=float)
        
        if len(errors) == 0:
            raise ValueError(f"No errors found for {city_name}, {country_code} on {date}. Cannot compute metrics.")
        elif len(errors) > 24:
            raise ValueError(f"More than 24 errors found for {city_name}, {country_code} on {date}. Found {len(errors)} errors. Data may be corrupted.")

        mae = float(np.mean(np.abs(errors)))
        rmse = float(np.sqrt(np.mean(errors**2)))
        
        bootstrap_mae, bootstrap_rmse = compute_bootstrap_distributions(errors, num_samples=num_samples)
        daily_errors[str(date)] = {
            'mae': mae,
            'rmse': rmse,
            'mae_dist': bootstrap_mae,
            'rmse_dist': bootstrap_rmse
        }

    return daily_errors

def analyze_all_cities_predictions(cities: list[tuple[str, str]], df: pd.DataFrame, output_file: str, num_samples: int) -> pd.DataFrame:
    rows = []
    for city_name, country_code in cities:
        bootstrap_results = compute_daily_bootstrap_metrics(df, city_name=city_name, country_code=country_code, num_samples=num_samples)
        for date, result in bootstrap_results.items():
            sample_mae, ci_lower_mae, ci_upper_mae = compute_confidence_interval(result['mae_dist'])
            sample_rmse, ci_lower_rmse, ci_upper_rmse = compute_confidence_interval(result['rmse_dist'])

            # Check if the sample MAE and RMSE are within 5% of the original MAE and RMSE, respectively
            if result['mae'] != 0 and (pct_dif := np.abs(sample_mae - result['mae']) / result['mae']) > 0.05:
                print(f"WARNING: Sample MAE ({sample_mae:.4f}) differs by {pct_dif*100:.2f}% from original MAE ({result['mae']:.4f}) for {city_name} on {date}.")
            if result['rmse'] != 0 and (pct_dif := np.abs(sample_rmse - result['rmse']) / result['rmse']) > 0.05:
                print(f"WARNING: Sample RMSE ({sample_rmse:.4f}) differs by {pct_dif*100:.2f}% from original RMSE ({result['rmse']:.4f}) for {city_name} on {date}.")

            rows.append({
                "city": city_name,
                "date": date,
                "mae": round(result['mae'], 2),
                "mae_ci": (round(ci_lower_mae, 1), round(ci_upper_mae, 1)),
                "rmse": round(result['rmse'], 2),
                "rmse_ci": (round(ci_lower_rmse, 1), round(ci_upper_rmse, 1)),
            })

    bootstrap_results_df = pd.DataFrame(rows)
    bootstrap_results_df.to_csv(f"data/{output_file}", index=False)
    return bootstrap_results_df

In [210]:
# Latency
analyze_all_cities_predictions(cities, latency_predictions, output_file="latency_bootstrap_results.csv", num_samples=1000000)


,city,date,mae,mae_ci,rmse,rmse_ci
0,New York,2025-11-24,8.86,"(6.3, 11.9)",11.32,"(7.7, 15.3)"
1,New York,2025-11-25,9.87,"(7.8, 11.9)",11.15,"(9.2, 12.9)"
2,New York,2025-11-26,7.35,"(5.3, 9.4)",8.93,"(6.9, 10.7)"
3,New York,2025-11-27,9.19,"(7.4, 11.0)",10.22,"(8.6, 11.7)"
4,New York,2025-11-28,8.69,"(6.6, 10.8)",10.14,"(8.0, 12.1)"
5,New York,2025-11-29,8.92,"(6.9, 10.9)",10.19,"(8.3, 12.0)"
6,New York,2025-11-30,7.14,"(5.2, 9.4)",8.92,"(6.1, 11.5)"
7,Santiago,2025-11-24,15.11,"(10.7, 19.8)",18.53,"(13.5, 23.4)"
8,Santiago,2025-11-25,19.25,"(15.4, 23.0)",21.10,"(17.6, 24.3)"
9,Santiago,2025-11-26,18.73,"(13.0, 25.0)",22.80,"(16.2, 29.2)"


In [211]:
# Throughput
analyze_all_cities_predictions(cities, throughput_predictions, output_file="throughput_bootstrap_results.csv", num_samples=1000000)


,city,date,mae,mae_ci,rmse,rmse_ci
0,New York,2025-11-24,41.73,"(28.3, 55.6)",49.26,"(35.2, 61.7)"
1,New York,2025-11-25,32.84,"(24.4, 41.7)",39.12,"(29.7, 47.5)"
2,New York,2025-11-26,32.49,"(22.4, 44.3)",42.51,"(27.3, 56.6)"
3,New York,2025-11-27,49.03,"(34.3, 65.4)",62.71,"(43.4, 80.0)"
4,New York,2025-11-28,56.10,"(38.6, 74.3)",71.76,"(53.4, 87.6)"
5,New York,2025-11-29,35.11,"(21.0, 51.4)",51.89,"(30.9, 69.9)"
6,New York,2025-11-30,36.45,"(22.4, 52.6)",50.90,"(30.6, 68.3)"
7,Santiago,2025-11-24,20.17,"(12.6, 28.3)",25.48,"(16.9, 33.1)"
8,Santiago,2025-11-25,20.42,"(14.1, 26.9)",25.30,"(19.1, 30.6)"
9,Santiago,2025-11-26,18.54,"(13.1, 24.4)",22.77,"(16.6, 28.4)"
